# 06.3 — Automated checks and updates

[Class index](README.md) · [Previous — 06.2](06.2-from-github-to-hosting.ipynb) · [Next — 06.4](06.4-guided-deployment-and-ci-practice.ipynb)

**Purpose:** Read supplied automated checks and verify a later change at the correct commit.

**Outcomes:** Explain workflow/job/step; run the same checks locally; inspect an Actions result; verify the public release independently.

**Prerequisites:** The complete class environment and saved files. Hosted access is needed for hosted evidence; the notebook's default inspection remains local.

**In class — 60–80 minutes teaching/demo; 80–90 exercise. Independent work — 10 minutes**, inside the 60 CI minutes.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
required = ["streamlit_app.py", "pyproject.toml", "uv.lock", "fixtures", "solutions"]
if not all((root / name).exists() for name in required):
    raise RuntimeError(
        "Open 06-deployment.code-workspace from the complete class folder; "
        "select that folder's .venv kernel and restart. Expected app/fixtures here."
    )
print("Working folder:", root)
print("Kernel:", sys.executable)

In [ ]:
from report_app.harness import run
from report_app.provider import make_replay

workflow = (root / ".github/workflows/checks.yml").read_text()
print(workflow)
result = run("Supplied repeated-call scenario", make_replay("repeated"))
assert result["outcome"] == "tool_limit"
assert result["tool_executions"] == 2
print("An offline behavior the supplied tests protect:", result["outcome"])

## Read the supplied workflow from top to bottom

**Continuous integration (CI)** runs automated checks when a change arrives. A **workflow** is the YAML file describing when and what to run. A **job** runs on a fresh **runner** machine; a **step** performs one operation within that job. You do not need to memorize YAML or write a pipeline from scratch.

The supplied file responds to pushes and pull requests. Its one job checks out the commit, installs uv, selects Python from `.python-version`, installs the locked environment, and runs tests, lint, and formatting checks. `contents: read` is enough to inspect this repository. No deployment key or model secret is needed. The action references are supplied; package/action installation needs network access, while the tests avoid external services.

`pytest` checks behavior: saved totals/absence, validation, actual execution limits, SDK JSON transport with a mock, and the real Streamlit script through headless AppTest. Ruff lint catches selected errors; `ruff format --check` reports formatting without changing files. Passing formatting cannot establish correct totals, and passing a mocked transport test cannot establish live-model quality.

[uv integration](https://docs.astral.sh/uv/guides/integration/github/), [Actions quickstart](https://docs.github.com/en/actions/get-started/quickstart).

## Local checks before a push

In the student repository terminal:

```sh
uv run --locked python -m pytest -q
uv run --locked ruff check streamlit_app.py report_app solutions tests
uv run --locked ruff format --check streamlit_app.py report_app solutions tests
```

If a check fails, read its first useful failure and fix the cause. Do not delete the assertion or remove the test step to make the check green. For an ordinary formatting change, run Ruff's formatter on the edited file, inspect the diff, and rerun the check. The required task is to run supplied checks and inspect a result; writing a test suite or another harness is not required.

Keep the workflow at the student repository root's `.github/workflows`. A nested downloadable example is not an active workflow in the course repository. If a pushed commit has no run, check placement and trigger before debugging the app.

### Visual guide — One push, two independent paths

<img src="images/independent-ci-and-hosting.png" alt="One GitHub push independently triggers an Actions check path and a hosting update path; their results are checked separately." width="552" style="max-width: 100%; height: auto; background-color: white;">

Actions and Community Cloud observe the repository through separate mechanisms. A passing check does not establish the hosted version, and a red check does not itself block automatic hosting updates. Verify the commit/run and visible release independently.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/independent-ci-and-hosting.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

## A passing check and a hosted version are different evidence

Change the supplied `RELEASE` constant to `06.1`; it appears as a public caption so a peer can recognize the update. Run checks, inspect/stage the diff, commit, and push manually. Record the full commit SHA using `git rev-parse HEAD`. Open **Actions → Checks** for that exact commit and inspect the test/lint/format steps.

Separately open the configured hosted app, wait for its update if needed, and verify **Release 06.1**. The host follows the chosen branch independently of Actions. This lesson does not install a gate that prevents auto-deployment after a failed check. A green Actions run is not proof the host loaded the new code; a changed page is not proof its tests passed.

This distinction makes incident diagnosis easier: wrong branch, old build, failing test, and broken runtime are different causes. Record the observed state instead of silently assuming both systems updated together.

## Exercise 06.3.E1 — Verify a subsequent release

> **In class — 10 minutes**
>
> **Goal:** Connect a one-line change to local checks and separate hosted evidence.
>
> **Start here:** RELEASE in streamlit_app.py; supplied workflow and test commands above.
>
> **Task:** Write the target release string, change that one line in the app, run supplied checks, and manually commit/push in your student repository. Inspect Actions and hosting independently.
>
> **Check:** Record the commit, check result/run URL, and public app release. Keep missing hosted evidence pending.
>
> **Hint:** Use RELEASE = "06.1"; the reference contains manual commands but never runs Git for you.

In [ ]:
# target_release = ...
# Commit:
# Actions run/result:
# Public URL and visible release:

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_06_3_e1.py

## Independent work — 10 minutes

Inspect your exact Actions run and its three checking steps. Compare them with local commands. Read [deployment guide section 4](DEPLOYMENT.md) before adapting the test command in 06.4. If hosting/Actions access is delayed, record that fact and inspect the supplied workflow text; do not label that inspection a passed hosted run.

**Completion check:** explain workflow/job/step, the tested commit, and why CI does not automatically gate the host. **Reflection:** which evidence would you collect if checks passed but the URL still displayed release 06.0?